# 🎬 OpenShorts no Google Colab (Aceleração por GPU / CUDA)

Este notebook executa o servidor **OpenShorts** utilizando a GPU do Google Colab (T4 / V100 / A100) para processamento ultra-rápido de transcrição (`faster-whisper`), rastreamento facial (`MediaPipe` / `YOLOv8`), renderização (`FFmpeg`) e publicação automática com os novos módulos de **Auto-Channel Watcher**, **Smart Scheduler** e conexão com a **Extensão Chrome 1-Click Shortify**.

### 1. Verificar GPU Disponível

In [ ]:
!nvidia-smi

### 2. Instalar Dependências de Sistema e Pacotes Python

In [ ]:
# Instalar pacotes de sistema
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-noto fonts-noto-cjk libgl1-mesa-glx pycloudflared

# Clonar ou atualizar o repositório OpenShorts
import os
if not os.path.exists('/content/openshorts'):
    !git clone https://github.com/diegodev/openshorts.git /content/openshorts
    %cd /content/openshorts
else:
    %cd /content/openshorts
    !git pull

# Instalar dependências Python otimizadas para GPU
!pip install -q -r requirements.txt
!pip install -q pycloudflared pyngrok uvicorn

### 3. Configurar Variáveis de Ambiente e Chaves de API
Insira sua chave da Gemini API (Google AI Studio) abaixo:

In [ ]:
import os
from google.colab import userdata

# Tenta obter de segredos do Colab ou define diretamente
try:
    gemini_key = userdata.get('GEMINI_API_KEY')
except Exception:
    gemini_key = 'SUA_GEMINI_API_KEY_AQUI'

os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['WHISPER_DEVICE'] = 'cuda'
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16'
os.environ['MAX_CONCURRENT_JOBS'] = '2'

print('Configuração salva com sucesso! GPU habilitada para Whisper & YOLO.')

### 4. Iniciar Túnel Público (Cloudflare Tunnel) & Servidor FastAPI
Gera uma URL pública `.trycloudflare.com` para conectar com a **Extensão Chrome 1-Click Shortify** e automações n8n/webhooks.

In [ ]:
from pycloudflared import try_cloudflare
import subprocess
import time

# Iniciar o Cloudflare Tunnel na porta 8000
tunnel_url = try_cloudflare(port=8000)
print('='*70)
print(f'🚀 URL PÚBLICA DA API DO OPENSHORTS: {tunnel_url.tunnel}')
print('Cole esta URL nas configurações da Extensão Chrome 1-Click Shortify!')
print('='*70)

# Iniciar o servidor FastAPI Backend
!python -m uvicorn app:app --host 0.0.0.0 --port 8000